# AgriVision AI · real classifier training

Phase 1: train MobileNetV3-Small on five original PlantVillage tomato categories.
No predictions or metrics are simulated. Run cells in order and download artifacts
before your free Colab session ends. Choose **Runtime → Change runtime type → T4 GPU**
if offered; CPU is supported. Eight epochs is a starting configuration, not a
promise of runtime or accuracy.

First create the upload package on your Mac from the repository root:
`python3.11 -m training.package_colab`.
Upload `artifacts/agrivision_colab_source.zip` in the next cell.

In [ ]:
import io
import os
import sys
import zipfile
from pathlib import Path

from google.colab import files

PROJECT_ROOT = Path("/content/agrivision-ai")
uploaded = files.upload()
if "agrivision_colab_source.zip" not in uploaded:
    raise ValueError("Select agrivision_colab_source.zip from the project artifacts folder.")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded["agrivision_colab_source.zip"])) as archive:
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        if not target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Unsafe source archive path.")
    archive.extractall(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
assert (PROJECT_ROOT / "training/train.py").is_file()
print("Source ready:", PROJECT_ROOT)

## Install shared dependencies

Install before importing Torch. If Colab requests a restart, restart the session
and rerun the source upload cell, then continue. The notebook uses the same pinned
Torch/Torchvision pair and preprocessing code as the local API.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT_ROOT / "training/requirements.txt"),
    ],
    check=True,
)

In [ ]:
import platform

import torch
import torchvision

print("Python:", platform.python_version())
print("Torch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU selected; expect a slower run.")

## Dataset source and licensing

We use [the original PlantVillage repository](https://github.com/spMohanty/PlantVillage-Dataset)
by Mohanty, Hughes, and Salathé, only five `raw/color` tomato folders plus grouping
metadata. Cite [the 2016 paper](https://doi.org/10.3389/fpls.2016.01419).
The author's [Hugging Face card](https://huggingface.co/datasets/mohanty/PlantVillage)
reports CC BY-SA 3.0; the upstream loader calls this an assumption. Verify terms for
your copy before redistribution. Our MIT code license does not cover dataset images.

The next cell downloads this documented source. For an existing local/Drive copy,
set `USE_EXISTING_DATASET = True` and edit the directory. Mount Drive first if needed:
`from google.colab import drive; drive.mount('/content/drive')`.

Known related leaves, recognizable filename copies, and exact decoded duplicates
stay together in all partitions. Unknown relationships and recompressed duplicates
need manual auditing; controlled-dataset accuracy does not establish field accuracy.

In [ ]:
import json

from backend.app.ml.classes import PLANTVILLAGE_NAMES

USE_EXISTING_DATASET = False
EXISTING_DATASET_ROOT = Path("/content/drive/MyDrive/PlantVillage")
DATASET_SOURCE = "https://github.com/spMohanty/PlantVillage-Dataset.git"

if USE_EXISTING_DATASET:
    DATASET_ROOT = EXISTING_DATASET_ROOT
    revision = None
else:
    DATASET_ROOT = Path("/content/PlantVillage-Dataset")
    if not DATASET_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--filter=blob:none",
                "--sparse",
                DATASET_SOURCE,
                str(DATASET_ROOT),
            ],
            check=True,
        )
    subprocess.run(
        [
            "git",
            "-C",
            str(DATASET_ROOT),
            "sparse-checkout",
            "set",
            *[f"raw/color/{name}" for name in PLANTVILLAGE_NAMES],
            "leaf_grouping",
        ],
        check=True,
    )
    revision = subprocess.check_output(
        ["git", "-C", str(DATASET_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
assert DATASET_ROOT.is_dir(), "Set an existing PlantVillage directory."
SOURCE_PROVENANCE = {
    "source_url": DATASET_SOURCE,
    "source_revision": revision,
    "user_supplied_copy": USE_EXISTING_DATASET,
    "dataset_variant": "color",
    "reported_license": "CC BY-SA 3.0 (author dataset card; verify upstream terms)",
    "citation": "Mohanty, Hughes, Salathe (2016), doi:10.3389/fpls.2016.01419",
}
print(json.dumps(SOURCE_PROVENANCE, indent=2))

## Train and evaluate

Defaults: fixed class indices; 70/15/15 stratification of indivisible groups; seed 42;
224×224 images; ImageNet normalization; train-only augmentation; frozen feature
parameters and feature BatchNorm; AdamW (lr 0.001, weight decay 0.0001); batch 32;
early stopping patience 3 on validation loss. Best weights are reloaded before
computing held-out accuracy, macro precision/recall/F1, per-class scores, and the
confusion matrix. Read `training/config.yaml` to configure other settings.

This cell prints image/group counts and actual epoch measurements. Choose a fresh
`RUN_DIR` for each rerun; existing checkpoints are protected. `LEAF_GROUPS = None`
auto-discovers the official map. Set a JSON path for custom metadata.

In [ ]:
EPOCHS = 8
RUN_DIR = Path("/content/agrivision_run")
LEAF_GROUPS = None  # Or Path('/content/drive/MyDrive/leaf_groups.json')

command = [
    sys.executable,
    "-m",
    "training.train",
    "--config",
    "training/config.yaml",
    "--dataset-root",
    str(DATASET_ROOT),
    "--epochs",
    str(EPOCHS),
    "--device",
    "auto",
    "--output-dir",
    str(RUN_DIR),
]
if LEAF_GROUPS is not None:
    command.extend(["--leaf-groups", str(LEAF_GROUPS)])
subprocess.run(command, check=True, cwd=PROJECT_ROOT)
(RUN_DIR / "dataset_source.json").write_text(json.dumps(SOURCE_PROVENANCE, indent=2))
print("Training and held-out evaluation completed:", RUN_DIR)

## Inspect the actual results

Do not claim successful training if the preceding cell failed. Inspect class support
and confusion, not accuracy alone. Confidence is a softmax score among five classes,
not diagnostic certainty. This model has no non-leaf or unknown-disease detector.

In [ ]:
from IPython.display import Image as DisplayImage
from IPython.display import display

for filename in ("dataset_report.json", "evaluation.json", "run_metadata.json"):
    print(filename)
    print((RUN_DIR / filename).read_text())
display(DisplayImage(filename=str(RUN_DIR / "training_curves.png")))
display(DisplayImage(filename=str(RUN_DIR / "confusion_matrix.png")))

## Export to your Mac

Run the next cell to download `agrivision_model_export.zip`. Extract it on your Mac
and copy **both** `best_model.pt` and `class_mapping.json` into the project's `models/`
folder. Preserve all evaluation and split artifacts for the academic report.

With your `.venv` active, from the project root run:
```
python -m backend.app.ml.inference path/to/leaf.jpg --model-dir models
python -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8000
```
Check `http://127.0.0.1:8000/health` for `model_loaded: true`. Restart the backend after
copying weights. Colab `/content` is temporary; download before disconnecting.

In [ ]:
from backend.app.ml.inference import InferenceEngine

EXPORT_FILES = (
    "best_model.pt",
    "class_mapping.json",
    "evaluation.json",
    "run_metadata.json",
    "history.json",
    "training_curves.png",
    "confusion_matrix.png",
    "split_manifest.json",
    "dataset_report.json",
    "dataset_source.json",
)
for filename in EXPORT_FILES:
    assert (RUN_DIR / filename).is_file(), (
        f"Missing {filename}; complete training/evaluation first."
    )
checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location="cpu", weights_only=True)
assert checkpoint["trained_epochs"] >= 1
engine = InferenceEngine(RUN_DIR)
engine.load()  # Confirm exported weights and mapping satisfy the serving contract.
EXPORT_PATH = Path("/content/agrivision_model_export.zip")
with zipfile.ZipFile(EXPORT_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in EXPORT_FILES:
        archive.write(RUN_DIR / filename, filename)
files.download(str(EXPORT_PATH))
print("Export ready:", EXPORT_PATH)